In [ ]:
# Load the spaCy model
## This model have the tags on it. One of them is 'ORG' so I choose it to extract the species.

nlp = spacy.load("en_ner_bionlp13cg_md") 

In [ ]:
# Function to extract species from the combined descriptions
# This function takes a text input, processes it with the spaCy model, and extracts named entities related to species.
# It returns a dictionary with the entity labels as keys and the corresponding entity texts as values.
# The function uses the spaCy model to identify named entities in the text and groups them by their entity labels.
# The output is a dictionary where the keys are the entity labels (e.g., 'ORG' for organisms) and the values are lists of entity texts that correspond to those labels.
# ------------------------------------------------------------  

def extract_species(text):
    doc= nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]

    dictionary_entities = {}
    for k,v in entities:
        if v not in dictionary_entities:
            dictionary_entities[v]=[]
        dictionary_entities[v].append(k)
    
    return dictionary_entities

In [ ]:
# Extracting species from the combined descriptions
# This will create a new column in the pick_headers_df dataframe with the extracted species from the combined descriptions.
# The extracted species will be stored in a dictionary with the entity type as the key
# and the list of species as the value.
# ------------------------------------------------------------

for row, item in dfs.iterrows():
    my_text = item['Description_combined']
    dic_entities = extract_species(my_text)

    dfs.at[row, 'Entities_SpaCy'] = str(dic_entities)

In [ ]:
# Creating a new column with the species extracted from the Entities_SpaCy column
# This column will be used to store the species extracted from the Entities_SpaCy column.
# It used ast to transform the string representation of the dictionary into a dictionary object.
# --------------------------------------------------------------

for row, item in dfs.iterrows():
    dataframe_row = dfs.loc[row, 'Entities_SpaCy']
    dictionary_entities = ast.literal_eval(dataframe_row)
    entities_keys = dictionary_entities.keys()

    if ('ORGANISM') in entities_keys:
        
        dfs.at[row, 'Species_SpaCy'] = dictionary_entities['ORGANISM'] if 'ORGANISM' in entities_keys else "Not found"
    else:
        dfs.at[row, 'Species_SpaCy'] = 'Not found'

In [ ]:
for i, item in dfs.iterrows():
    list_species_spacy = item['Species_SpaCy']
    list_species_scientific_name = []

    for x in list_species_spacy:
        values = x.split(' ')
        #print(values)
        for unique_value in values:
            #print(unique_value)
            
            url = f'https://www.ebi.ac.uk/ena/taxonomy/rest/any-name/{unique_value}'
            #print(url)

            response = requests.get(url)
        
            if response.status_code == 200:
                data = response.json()
                #print(data)
                if data:
                    scientific_name = data[0]['scientificName']
                    #print("found")

                    list_species_scientific_name.append(scientific_name)
                    print(f"Scientific name for {x}: {scientific_name}")
                else:
                    scientific_name = 'Not found'
                    print(f"Scientific name for {x}: {scientific_name}")

    dfs.at[i, 'Species_Scientific_Name_SciSpacy'] = str(list_species_scientific_name)

In [ ]:
for i, item in dfs.iterrows():
    if isinstance(item['Study_Organism'], str) and item['Study_Organism'].strip() != '':
        real_specie = item['Study_Organism']
        scispacy_specie = item['Species_Scientific_Name_SciSpacy']

        if real_specie in scispacy_specie:

            print('The real species is in SciSpacy')
            dfs.at[i, 'Specie_found_Scispacy'] = 'Yes'
        else:
            print('The real species is NOT in SciSpacy')
            dfs.at[i, 'Specie_found_Scispacy'] = 'No'
    else:
        print('No real species to check in SciSpacy')
        dfs.at[i, 'Specie_found_Scispacy'] = 'No species provided'